In [ ]:
%xmode Context
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt

import sys
from pathlib import Path

# help locating the package sgpykit (comment this out if you installed it)
PKG_PARENT = Path().resolve().parent
sys.path.insert(0, str(PKG_PARENT))

import sgpykit as sg

# Set up logging with sgpykit convenience functions
sg.set_logger_basic_format()
sg.set_logger_info_level()
# For more verbose output, use: sg.set_logger_debug_level()
# For custom format, use: sg.set_logger_custom_format(format)

# PART 3: Integration on Sparse Grids - Basics

In this part, we show how to use the Kit to perform high-dimensional quadrature. We consider the
following function, for which we know the analytic expression of the integral:

$$
f(\mathbf{x}) = \prod_{i=1}^N \frac{1}{\sqrt{x_i + b}}, \quad \mathbf{x} \in [-1,1]^N
$$

In [ ]:
#f = lambda x,b: np.prod(1.0 / np.sqrt(x + b), axis=0, keepdims=True)
f = lambda x, b: np.prod((x + b)**-0.5, axis=0)

b = 3
N = 4
I_1d = (2*np.sqrt(1+b)-2*np.sqrt(-1+b))
I_ex = I_1d**N

# generate the knots and the SM grid. 'nonprob' means we are integrating w.r.t. the pdf rho(x)=1 and not rho(x)=1/prod(b_i - a_i)
knots = lambda n: sg.knots_CC(n,-1,1,'nonprob')
w = 4
S,_ = sg.create_sparse_grid(N,w,knots,sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

In [ ]:
# compute integral
mknots = Sr.knots
mweights = np.transpose(Sr.weights)

I = f(mknots, b) @ mweights
assert np.isclose(I,1.883984044753591), "Results mismatch"

In [ ]:
# alternatively use
I2,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr) # Sr must be reduced here

print('difference between values:', I-I2)

In [ ]:
# compare with exact value
print('quad error:')
np.abs(I-I_ex)

Sometimes, we have access to the evaluations of $f$ from earlier code; then we just need to do the linear combination. The package provides
a convenience wrapper for this purpose, instead of typing `f_vals @ np.transpose(Sr.weights)`.

In [ ]:
print('convenience wrapper')

f_vals = f(Sr.knots, b)
I3,_ = sg.quadrature_on_sparse_grid(f_vals, S=None, Sr=Sr)
I2-I3

The convenience wrapper can also handle the case of computing quadrature for multiple functions at the same time. The values of each function
must be stored as rows of a matrix.

In [ ]:
many_f = np.vstack([f_vals, f_vals, f_vals, f_vals, f_vals])

I4,_ = sg.quadrature_on_sparse_grid(many_f,S=None, Sr=Sr)
I4

## Integration on Tensor Grids

In [ ]:
# f = ... see above
b = 3
N = 2
I_1d = (2*np.sqrt(1+b)-2*np.sqrt(-1+b))
I_ex = I_1d**N

In [ ]:
# let's build a tensor grid with the following choices
knots = lambda n: sg.knots_CC(n,-1,1,'nonprob')
ii = np.array([6,5])

m = sg.lev2knots_doubling(ii)
T = sg.tensor_grid(N, m, knots)
S = sg.tensor_to_sparse(T)

# with the call above, S.idx is automatically set to the number of points in each dir,
# which implies S.idx = lev2knots(ii). If you want S.idx = ii, use the following call
# S = tensor_to_sparse(T,ii);

Sr = sg.reduce_sparse_grid(S)
sg.is_sparse_grid(S)

In [ ]:
I,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)
# compare with exact value
print('quad error:')
np.abs(I-I_ex)

## Use Other Quadrature Knots
As already seen in the introduction, other quadrature knots are available.

In [ ]:
# f = ... see above
b = 3
N = 4
I_1d = (2*np.sqrt(1+b)-2*np.sqrt(-1+b))
I_ex = I_1d**N

In [ ]:
# let's build a tensor grid with the following choices
knots = lambda n: sg.knots_uniform(n,-1,1,'nonprob') # <- now uniform
w = 4
S,_ = sg.create_sparse_grid(N,w,knots,sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

In [ ]:
I,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)
# compare with exact value
print('quad error:')
np.abs(I-I_ex)

## Modify Quadrature Domain
Suppose integrating over $(-1,3)^N$.

In [ ]:
# f = ... see above
b = 3
N = 4
I_1d = (2*np.sqrt(3+b)-2*np.sqrt(-1+b))  # <- 3 instead of 1
I_ex = I_1d**N

In [ ]:
# generate the knots in (-1,3)
knots = lambda n: sg.knots_CC(n,-1,3,'nonprob')
w = 6
S,_ = sg.create_sparse_grid(N,w,knots,sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

In [ ]:
I,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)
# compare with exact value
print('quad error:')
np.abs(I-I_ex)

## Compute Moments of Random Variables
Here we compute $E[f(\mathbf{x})] = \int_{[-2, 1]\times[0.5, 6]} f(\mathbf{x}) /(3\cdot 5.5) \, d\mathbf{x}$, where $3\cdot 5.5$ is the size of the domain.

In [ ]:
# f = ... see above
b = 3
N = 2
I_ex = 1/3/5.5*(2*np.sqrt(1+b)-2*np.sqrt(-2+b))*(2*np.sqrt(6+b)-2*np.sqrt(0.5+b))

In [ ]:
# the best-practice is to generate knots on (-2,1) and (0.5,6), specifying 'prob' as input to the
# knots-generatic function
knots1=lambda n: sg.knots_CC(n,-2,1,'prob')  # knots1=@(n) knots_CC(n,-2,1); would work as well as 'prob' is the default value
knots2=lambda n: sg.knots_CC(n,0.5,6,'prob') # knots2=@(n) knots_CC(n,0.5,6); would work as well as 'prob' is the default value

In [ ]:
w = 6
S,_ = sg.create_sparse_grid(N, w, [knots1, knots2], sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)
I,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)
# compare with exact value
print('quad error:')
np.abs(I-I_ex)

## Recycle Evaluations from Previously Computed Grids

Just as [`evaluate_on_sparse_grid()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.main.html#sgpykit.main.evaluate_on_sparse_grid), [`quadrature_on_sparse_grid()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.main.html#sgpykit.main.quadrature_on_sparse_grid) provides evaluation recycling.

In [ ]:
f = lambda x,b: np.prod(1./np.sqrt(x+b), axis=0)
b = 5
N = 2
# the starting grid
w = 7
knots = lambda n: sg.knots_CC(n,-2,1,'prob')
S,_ = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)
IS,evals_S = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)

In [ ]:
# the new grid
w = 8
T,_ = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)
Tr = sg.reduce_sparse_grid(T)
# the recycling call.
IT_rec,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=T, Sr=Tr, evals_old=evals_S, S_old=S, Sr_old=Sr)

np.testing.assert_almost_equal(IT_rec, IS)

In [ ]:
# the non-recycling call
IT,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Tr)

np.testing.assert_almost_equal([IT_rec, IT], 0.228763833671747)

## How to Build More Complex Sparse Grids: Anisotropic Grids

In [ ]:
# f = ... see above
b = 3
N = 4
I_1d = (2*np.sqrt(1+b)-2*np.sqrt(-1+b))
I_ex = I_1d**N

In [ ]:
# specify a rule like in Back Nobile Tamellini Tempone, `Stochastic Spectral Galerkin and Collocation...a  numerical comparison''
rates = [1, 2, 2, 2]
knots = lambda n: sg.knots_uniform(n,-1,1,'nonprob')
lev2nodes,idxset = sg.define_functions_for_rule('TD', rates)
w = 4
S2,_ = sg.create_sparse_grid(N,w,knots,lev2nodes,idxset)
Sr = sg.reduce_sparse_grid(S2)
I,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)
# compare with exact value
print('quad error:')
np.abs(I-I_ex)  # matlab result: 1.456974987823489e-04

## How to Build More Complex Sparse Grids: Use multiidx_box_set
As seen in the introduction, specify directly the set of multi-indices involved.
Here, we generate the box set of all multi-indices $\leq [3, 5, 2, 3]$ in lexicographic order.

In [ ]:
# f = ... see above
b = 3
N = 4
I_1d = (2*np.sqrt(1+b)-2*np.sqrt(-1+b))
I_ex = I_1d**N

C,_ = sg.multiidx_box_set([2, 4, 1, 2])  # X is C without [2 4 1 2]
knots = lambda n: sg.knots_uniform(n,-1,1,'nonprob')
S3,_ = sg.create_sparse_grid_multiidx_set(C, knots, sg.lev2knots_lin)
Sr = sg.reduce_sparse_grid(S3)
I,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)
# compare with exact value
print('quad error:')
np.abs(I-I_ex)  # matlab result: 6.552113139615123e-04

## Convergence Study

See test_sparse_quadrature.m (not ported to Python yet).

## NEXT: 

=> **[Go to tutorial "Interpolation"](03-interpolation.ipynb)**